In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [2]:
# Extracting temporal features
def toTime(df):
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    df['month'] = df['time'].dt.month
    df['day'] = df['time'].dt.day
    df['hour'] = df['time'].dt.hour
    df['minute'] = df['time'].dt.minute
    df['second'] = df['time'].dt.second
    df['dayofweek'] = df['time'].dt.dayofweek  # Monday=0, Sunday=6
    df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)
    return df

# Create a DataFrame for submission
def submit(test_ids, ylong, ylat):
    # Create a submission DataFrame
    submission = pd.DataFrame({
        'ID': test_ids,  # Pass the test ID column
        'longitude_predicted': ylong,  # Your predicted longitudes
        'latitude_predicted': ylat    # Your predicted latitudes
    })

    # Ensure the correct column order
    submission = submission[['ID', 'longitude_predicted', 'latitude_predicted']].sort_values(by = 'ID').reset_index(drop=True)

    # Find the next available filename by checking existing files
    i = 1
    while os.path.exists(f'jakobs_results_{i}.csv'):
        i += 1

    # Save to CSV with the next available filename
    filename = f'jakobs_results_{i}.csv'
    submission.to_csv(filename, index=False)
    print(f"Results saved to {filename}")

# Function to split each vessel's data
def time_series_split_by_vessel(df, train_size=0.97):
    train_dfs = []
    val_dfs = []

    # Group by vesselId to split each vessel's data individually
    for vessel_id, vessel_data in df.groupby('vesselId'):
        # Calculate the split index for this vessel
        split_index = int(len(vessel_data) * train_size)

        # Split into training and validation sets for this vessel
        train_dfs.append(vessel_data.iloc[:split_index])
        val_dfs.append(vessel_data.iloc[split_index:])

    # Concatenate all vessel's training and validation data back into a DataFrame
    train_df = pd.concat(train_dfs)
    val_df = pd.concat(val_dfs)

    return train_df, val_df

# Define the Haversine function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

In [3]:
# First we read the necessary csv files. NOTE: Different delimiter for some files.
df_train = pd.read_csv('/Users/jakobrudeovstaas/Desktop/ML/ais_train.csv', delimiter = '|')
df_test = pd.read_csv('/Users/jakobrudeovstaas/Desktop/ML/ais_test.csv')
df_vessels = pd.read_csv('/Users/jakobrudeovstaas/Desktop/ML/vessels.csv', delimiter = '|')
df_schedules = pd.read_csv('/Users/jakobrudeovstaas/Desktop/ML/schedules_to_may_2024.csv', delimiter = '|')
df_ports = pd.read_csv('/Users/jakobrudeovstaas/Desktop/ML/ports.csv', delimiter = '|')
# NOTE: Train is sorted by vessel ID and time for model to notice temporal trend.
df_train = df_train.sort_values(by = ['vesselId', 'time'])
df_test = df_test.sort_values(by = ['vesselId', 'time'])

In [4]:
# Extracting temporal features
df_train = toTime(df_train)
df_test = toTime(df_test)

In [5]:
# Calculate means of the columns without outliers
sog_mean = df_train.loc[df_train['sog'] < 30, 'sog'].mean()
cog_mean = df_train.loc[df_train['cog'] < 360, 'cog'].mean()
heading_mean = df_train.loc[df_train['heading'] < 360, 'heading'].mean()

# Cleaning the data. Ensuring all data is within reasonable limits and replacing outliers/erranous data with NaN
df_train['cog'] = df_train['cog'].apply(lambda x: float(x) if float(x) < 360 else cog_mean)
# NOTE: We determine that SOG above 25 knots shall be disregarded.
df_train['sog'] = df_train['sog'].apply(lambda x: float(x) if float(x) < 30 else sog_mean)
df_train['heading'] = df_train['heading'].apply(lambda x: float(x) if float(x) < 360 else heading_mean)
# NOTE: Navstat of 0 and 8 both indicate a moving vessel. We therefore combine these two
df_train['navstat'] = df_train['navstat'].replace(8, 0)

In [6]:
# Create lagging features for latitude, longitude, sog, cog, and navstat
for lag in range(1, 4):  # Adding lag-1, lag-2, and lag-3 features
    df_train[f'latitude_lag{lag}'] = df_train.groupby('vesselId')['latitude'].shift(lag)
    df_train[f'longitude_lag{lag}'] = df_train.groupby('vesselId')['longitude'].shift(lag)
    df_train[f'sog_lag{lag}'] = df_train.groupby('vesselId')['sog'].shift(lag)
    df_train[f'cog_lag{lag}'] = df_train.groupby('vesselId')['cog'].shift(lag)
    df_train[f'heading_lag{lag}'] = df_train.groupby('vesselId')['heading'].shift(lag)
    df_train[f'navstat_lag{lag}'] = df_train.groupby('vesselId')['navstat'].shift(lag)


last_known_lags = df_train.groupby('vesselId').last().reset_index()

# Select only the lagged columns and vesselId for merging with the test set
lagged_columns = [col for col in last_known_lags.columns if 'lag' in col] + ['vesselId']
last_known_lags = last_known_lags[lagged_columns]

df_test = df_test.merge(last_known_lags, on='vesselId', how='left')

# Fill missing values in lagged features with 0, mean, or another reasonable choice
for col in lagged_columns:
    df_test[col] = df_test[col].fillna(0)
    df_train[col] = df_train[col].fillna(0)

In [7]:
df_test.isna().any()[0:50]

ID                False
vesselId          False
time              False
scaling_factor    False
month             False
day               False
hour              False
minute            False
second            False
dayofweek         False
is_weekend        False
latitude_lag1     False
longitude_lag1    False
sog_lag1          False
cog_lag1          False
heading_lag1      False
navstat_lag1      False
latitude_lag2     False
longitude_lag2    False
sog_lag2          False
cog_lag2          False
heading_lag2      False
navstat_lag2      False
latitude_lag3     False
longitude_lag3    False
sog_lag3          False
cog_lag3          False
heading_lag3      False
navstat_lag3      False
dtype: bool

In [8]:
# Adding rolling averages
window_size = 3

df_train['sog_rolling_avg'] = df_train.groupby('vesselId')['sog'].transform(lambda x: x.rolling(window = window_size).mean())
df_train['cog_rolling_avg'] = df_train.groupby('vesselId')['cog'].transform(lambda x: x.rolling(window = window_size).mean())

# Extract the last rolling averages from the training data for each vesselId
last_rolling_avgs = df_train.groupby('vesselId').last().reset_index()

# Select only the columns you need for merging with the test set
last_rolling_avgs = last_rolling_avgs[['vesselId', 'sog_rolling_avg', 'cog_rolling_avg']]

# Merge the rolling averages into the test set based on vesselId
df_test = df_test.merge(last_rolling_avgs, on='vesselId', how='left')

In [9]:
# Adding time difference between timesteps
df_train['delta_time_min'] = df_train.groupby('vesselId')['time'].diff().dt.total_seconds() / 60
df_test['delta_time_min'] = df_test.groupby('vesselId')['time'].diff().dt.total_seconds() / 60

In [10]:
# Adding the avg SOG for each vessel ID. NOTE: mean is based on when the vessel is moving
isMoving = df_train[df_train['navstat'] == 0]
avg_sog_moving = isMoving.groupby('vesselId')['sog'].mean()
df_train['avg_sog_moving'] = df_train['vesselId'].map(avg_sog_moving)
df_test['avg_sog_moving'] = df_test['vesselId'].map(avg_sog_moving)

# Encoding labels
label_encoder = LabelEncoder()
df_train['vesselId_encoded'] = label_encoder.fit_transform(df_train['vesselId'])
df_test['vesselId_encoded'] = label_encoder.transform(df_test['vesselId'])

# Adding gross tonnage and length
df_train = pd.merge(df_train, df_vessels[['vesselId', 'GT', 'length']], on = 'vesselId', how = 'left')
df_test = pd.merge(df_test, df_vessels[['vesselId', 'GT', 'length']], on = 'vesselId', how = 'left')

In [ ]:
# Implementing a RandomForestRegressor
# First defining training, test sets and target predictors.
features = ['vesselId_encoded', 'day', 'month', 
            'latitude_lag1', 'longitude_lag1', 'sog_lag1', 'cog_lag1']

print(f"Features: {features}")

train_df, val_df = train_test_split(df_train, test_size=0.2, random_state=42)
X_train = train_df[features].copy()
X_val = val_df[features].copy()

y_train = train_df[['latitude', 'longitude']].copy()
y_val_lat = val_df['latitude'].copy()
y_val_long = val_df['longitude'].copy()

RFR = RandomForestRegressor(n_estimators = 100, n_jobs = -1, random_state = 42)

multi_output_regressor = MultiOutputRegressor(RFR)

multi_output_regressor.fit(X_train, y_train)

y_pred = multi_output_regressor.predict(X_val)

y_pred_lat = y_pred[:, 0]
y_pred_long = y_pred[:, 1]

mae_lat = mean_absolute_error(y_val_lat, y_pred_lat)
mae_long = mean_absolute_error(y_val_long, y_pred_long)
r2_lat = r2_score(y_val_lat, y_pred_lat)
r2_long = r2_score(y_val_long, y_pred_long)

print(f'Latitude: MAE: {mae_lat:.2f}, R^2: {r2_lat:.2f}')
print(f'Longitude: MAE: {mae_long:.2f}, R^2: {r2_long:.2f}')

In [ ]:
# Find common columns between df_train and df_test
common_columns = df_train.columns.intersection(df_test.columns)

# Print the common columns
print("Common columns in train and test datasets:")
print(common_columns)

In [ ]:
# Implementing a RandomForestRegressor
# First defining training, test sets and target predictors.
features = ['vesselId_encoded', 'day', 'month', 'length', 'GT', 
            'latitude_lag1', 'longitude_lag1', 'latitude_lag2', 'longitude_lag2', 
            'sog_lag1', 'cog_lag1']

X_train = df_train[features].copy()
X_test = df_test[features].copy()

y_train = df_train[['latitude', 'longitude']].copy()

RFR = RandomForestRegressor(n_estimators = 10, n_jobs = -1, random_state = 42, verbose = 2)
multi_output_regressor = MultiOutputRegressor(RFR)

multi_output_regressor.fit(X_train, y_train)

y_pred = multi_output_regressor.predict(X_test)

y_pred_lat = y_pred[:, 0]
y_pred_long = y_pred[:, 1]

# Extract feature importances for each target
latitude_importances = multi_output_regressor.estimators_[0].feature_importances_
longitude_importances = multi_output_regressor.estimators_[1].feature_importances_

# Average feature importances across targets for a combined importance
average_importances = np.mean([latitude_importances, longitude_importances], axis=0)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, average_importances, color="skyblue")
plt.xlabel("Feature Importance")
plt.title("Average Feature Importance for Latitude and Longitude Prediction")
plt.show()

submit(df_test['ID'], y_pred_long, y_pred_lat)

**Implementing a LSTM model. Which is more likely to perform well on temporal data.**

In [11]:
# Define categorical and continuous features separately
categorical_features = ['vesselId_encoded']
continuous_features = ['day', 'month', 'hour', 'dayofweek',
                       'length', 'GT',
                       'sog_lag1', 'cog_lag1',
                       'sog_lag2', 'cog_lag2',
                       'sog_lag3', 'cog_lag3',
                       'latitude_lag1', 'longitude_lag1', 
                       'latitude_lag2', 'longitude_lag2',
                       'latitude_lag3', 'longitude_lag3']

# Separate features for train and test
X_train_cat = df_train[categorical_features].copy()
X_train_cont = df_train[continuous_features].copy()
y_train = np.array(df_train[['latitude', 'longitude']].copy())

X_test_cat = df_test[categorical_features].copy()
X_test_cont = df_test[continuous_features].copy()

# Standardize only the continuous features
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(X_train_cont)
X_test_cont = scaler.transform(X_test_cont)

# Recombine scaled continuous features with unscaled categorical features
X_train = np.hstack([X_train_cat, X_train_cont])  # Concatenate categorical and scaled continuous features
X_test = np.hstack([X_test_cat, X_test_cont])

In [ ]:
def buildNN(inputDim):
    model = Sequential()
    model.add(Input(shape=(inputDim,)))
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.3))

    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))

    model.add(Dense(32, activation='relu'))
    model.add(Dense(2, activation='linear'))  # Output layer for latitude & longitude

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Define callbacks for early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)

inputDim = X_train.shape[1]

# Define model
model = buildNN(inputDim)

# Train model
history = model.fit(
    X_train, y_train,
    epochs=100,  # Higher epochs; early stopping will prevent overfitting
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr]
)

y_pred = model.predict(X_test)
y_pred_lat = y_pred[:, 0]
y_pred_long = y_pred[:, 1]

submit(df_test['ID'], y_pred_long, y_pred_lat)

Epoch 1/100
38052/38052 ━━━━━━━━━━━━━━━━━━━━ 36s 933us/step - loss: 237.5420 - mae: 9.5339 - val_loss: 166.6556 - val_mae: 8.4994 - learning_rate: 0.0010
Epoch 2/100
 5324/38052 ━━━━━━━━━━━━━━━━━━━━ 28s 871us/step - loss: 93.1912 - mae: 5.8558